# Dunnhumby seed43 — 최신 M2의 N/V 순위 기여 진단
**추가 학습 0회.** 공유 N/V L2=.0001인 M2의 기존 선택 **100 epoch** 가중치 한 개만 읽습니다. M3·M4·M5·H&M·추가 시드를 실행하지 않고, 기존 결과/가중치를 덮어쓰지 않습니다.

같은 공동학습 M2를 `ID only / ID+N / ID+V / full` 네 내부 방식으로 재평가합니다. N/V를 가리는 것은 진단이지 축을 제거한 새 제안모형이 아닙니다. **ID only는 M1이 아닙니다.** N/V는 곱셈형이라 두 축의 효과를 단순 합산할 수 없습니다. M1은 원본 집계지표만 참조하며 가중치를 새로 학습하지 않습니다.

학습≤683 / 반복 노출 개발684–690, 신규 상품 과업·학습쌍 제외·MIN_ITEM_INTER=1 유지. 원본 142개 지표를 재현하지 못하면 판독을 중단합니다. Top-10·20·50 정답 진입/이탈을 전체·저/중/고CLV로 기록합니다. 별도로 고정 학습표본 최대8192행에서 양성의 구매자 기여 제외(LOO) 입력과 전체 집계 입력 차이를 측정하며, 이것은 신규상품 성과나 실패 원인의 인과적 증명이 아닙니다. LOO 제거를 해결책으로 적용하지 않습니다.

최종 test·holdout·epoch 재선택 없음. 단일 개발시드의 사후 진단으로 유의성·일반화·CLV 귀속을 주장하지 않습니다. 기존 진단 함수를 재사용하며 새 Excel은 만들지 않습니다. 로컬 CPU 집중 검사 6개와 실제 Dunnhumby 입력·특징 해시 일치 검사를 통과했습니다. 실제 Drive 가중치 재평가는 아직 수행하지 않았습니다.

## 1. 원본 결과·선택 가중치 확인
GPU 런타임을 권장하지만 학습은 없습니다. Drive 연결이 끊어졌으면 계정/연결을 확인하고 이 셀부터 다시 실행하세요. checkpoint가 없거나 해시가 다르면 멈추며, 최신 epoch 가중치로 대체하지 않습니다.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
from google.colab import drive
ROOT = Path('/content/drive/MyDrive/논문/data')
REPORT = ROOT/'results_v3_dunnhumby_nv_modulated_id_l2_1e4_seed43_v1/reports/result.json'
CHECKPOINT = None  # 원래 가중치를 이동한 경우에만 정확한 경로 지정. SHA가 같아야 합니다.
OUT = ROOT/'results_v3_dunnhumby_nv_modulated_id_l2_1e4_seed43_checkpoint_diagnostic_v1'
try:
    if not ROOT.is_dir():
        drive.mount('/content/drive')
    if not REPORT.is_file():
        raise FileNotFoundError(f'완료 결과 없음: {REPORT}. REPORT에 원본 JSON 또는 결과 ZIP 경로를 지정하세요.')
    REPORT.open('rb').close()
except (OSError, ValueError, NotImplementedError) as exc:
    raise RuntimeError('Drive 연결 또는 결과 파일 읽기 실패. 원인을 확인하고 재실행하세요. 자동 재학습 없음.') from exc
SOURCE_COMMIT = '453d37612228f9da6ec82e33d4b4ed0e18935933'
REPO = Path('/content/clv-nv-checkpoint-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
if 'lightgcn_clv_v3' in sys.modules:
    assert Path(sys.modules['lightgcn_clv_v3'].__file__).resolve().parent == REPO.resolve(), '다른 실험 모듈이 로드되었습니다. 런타임을 다시 시작하세요.'
os.chdir(REPO)
sys.path.insert(0, str(REPO))
import clv_nv_modulated_checkpoint_diagnostic as audit
import pandas as pd
report, cfg, arm, checkpoint = audit.preflight(REPORT, CHECKPOINT)
print('확인:', arm['model_id'], '| 선택 epoch', arm['selected_epoch'])
print('가중치:', checkpoint)
print('추가 학습 0회 / M2 한 개의 내부 재평가 4방식')

## 2. 동일 개발자료 준비 — 학습 없음
학습 당시 코드·입력·N/V 특징 해시와 일치해야 다음 단계로 갑니다. 기존 파이프라인의 `Test` 출력은 이미 노출된 개발684–690의 구형 명칭이며 최종 test를 열지 않습니다.

In [ ]:
context = audit.prepare(REPORT, OUT, CHECKPOINT)
print('준비 완료. 다음 셀은 저장 모델 재평가만 수행합니다.')

## 3. 저장 모델 진단
원본 전체 성과 재현, 축별 마스킹, 정답 이동 합계 대조, 학습 입력 차이를 계산합니다. `full`은 원래 점수이고 나머지는 같은 M2의 내부 진단입니다.

In [ ]:
paths = audit.run(context)
print(json.dumps(paths, ensure_ascii=False, indent=2))

## 4. 핵심 표와 전체 원본 ZIP
여기서는 새 성공/실패 판정을 자동으로 내리지 않습니다. `full−id_only`, `full−id_n`, `full−id_v`와 정답 진입·이탈을 함께 봅니다. 전체·세그먼트·노출 지표와 학습표본 원본은 ZIP에 보존합니다. 50위 밖 정답 순위는 정확한 순위가 아니라 `>50` 구간입니다.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
result = json.loads((OUT/'diagnostic.json').read_text())
assert result['code_version'] == audit.VERSION and not result['new_training']
paths = result['paths']
absolute = pd.read_csv(paths['absolute'])
metrics = ['recall','ndcg','price_purchase_amount_weighted_hit','vndcg']
main = absolute[(absolute.segment == '전체') & absolute.metric.isin(metrics)]
print('전체 핵심 성과 — 같은 M2의 내부 진단, ID only는 M1이 아님')
print(main.pivot(index=['cutoff','metric'], columns='component', values='value').reindex(columns=audit.VIEWS).to_string())
print('정답 유입·이탈 — 전체 및 모든 CLV 구간')
print(pd.read_csv(paths['movement_summary']).to_string(index=False))
print('학습 양성의 LOO/전체 입력 차이 — 성과나 원인 귀속이 아님')
print(pd.read_csv(paths['train_input_shift_summary']).to_string(index=False))
print('원본 재현 지표 수:', result['original_metric_count'], '| 진단 시간(초):', round(result['elapsed_seconds'],1))
zip_path = Path('/content/nv_modulated_m2_checkpoint_seed43_diagnostic.zip')
with ZipFile(zip_path, 'w', compression=ZIP_DEFLATED) as archive:
    for name, path in paths.items():
        archive.write(path, arcname=Path(path).name)
print('결과:', zip_path)
files.download(str(zip_path))